# GSPC Measurement Engine — verified archive replay

This version repairs the notebook's archive-verification journey. It checks the **actual feed rows**, their ordered timestamp/leaf bindings, all declared counts, hash chain, Merkle root and mandatory Ed25519 signature before analysing anything. The source bytes and lane public key are fixed to an immutable Hugging Face revision.

**Data window:** 3–9 November 2026; retained manifest produced 10 November 2026. Running today does not make those observations current. No model, GPU, training, private runtime or provider API is used.

The first stage runs 39 offline controls, including altered feed bytes, substituted keys, malformed input and missing cryptography. A changed in-memory copy of the real feed is then rejected. Missing libraries or unreachable inputs stop execution; they are not a green result.

The lane key is a repository-history pin, not an independently established organisational identity. Measurement, not certification. Original signed records are retained unchanged.

Data attribution: GSPC live pulse — Council of AI, CC-BY-4.0 OR CC0-1.0. New verification code: MIT; unchanged evidence-event helper: Apache-2.0. The canonical evidence-event builder is reused unchanged from the Council of AI repository.


## 1. Load the exact embedded sources
No executable code is downloaded at runtime. Each embedded module is checked before execution. These checks detect accidental changes against this notebook's source lock; an independently pinned notebook/source revision is still needed to authenticate the lock itself.

In [ ]:
import hashlib, io, json, platform, sys, types, unittest
from pathlib import Path

def install_source(name, source, expected_sha):
    raw = source.encode("utf-8")
    if hashlib.sha256(raw).hexdigest() != expected_sha:
        raise RuntimeError("EMBEDDED_SOURCE_MISMATCH: " + name)
    module = types.ModuleType(name)
    module.__file__ = name + ".py"
    sys.modules[name] = module
    exec(compile(source, module.__file__, "exec"), module.__dict__)
    return module

pulse_replay = install_source('pulse_replay', '#!/usr/bin/env python3\n# SPDX-License-Identifier: MIT\n"""Replay one immutable public pulse snapshot; never collect or sign a new pulse.\n\nThe lane\'s published JSON canonicalisation and Merkle rule are preserved.\nA pinned key is a source-history anchor, not an independently proved identity.\n"""\nimport hashlib\nimport hmac\nimport json\nimport re\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nfrom datetime import datetime, timezone\n\nVERSION = "1.0.0"\nREPO = "csoai/evidence-index"\nREVISION = "60342db01ef9d65243ff4177b872382d4f12b87b"\nPREFIX = "gspc-pulse/2026-11-04"\nBASE = f"https://huggingface.co/datasets/{REPO}/resolve/{REVISION}/{PREFIX}/"\nPINNED_KEY = "4a50f0d758fbd7c7770fea7fd3fd69b14f94ee0b4b6a2833a15f008b0185f54a"\nGENESIS = b"pulse-series-genesis-2026-11-04"\nPINS = {\n    "feed.json": ("9598d5c78f43301a01b5da31e6014d5e95aaa708900d594dc185ef0bd3896f8c", 262144),\n    "pulse-freeze-manifest.json": ("0354e0b00c22aa765f6509f80888332a901198b28584df15a22b10038c19f643", 8192),\n    "pulse-freeze-manifest.sig": ("b7a964ac688cbb4b73a52e41ab0fbebcb72894286a80c90c95fe5289ff0f2fb8", 256),\n    "pulse-leaves.json": ("fa4fe0c25784511e67fb6cc049403d865f167d0c0c93acda05f77040b4f830b1", 65536),\n}\nHEX = re.compile(r"[0-9a-f]{64}\\Z")\nUTC = re.compile(r"\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}Z\\Z")\n\nclass VerificationError(ValueError):\n    """A checked invariant failed; callers must not analyse the unverified feed."""\n\nclass Uncheckable(RuntimeError):\n    """A required dependency or source is unavailable; never a successful check."""\n\ndef require(condition, reason):\n    if not condition:\n        raise VerificationError(reason)\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\ndef canonical(value):\n    # Exact published pulse rule, not JCS and not the GSPC card canonicalisation.\n    return json.dumps(value, sort_keys=True, separators=(",", ":"),\n                      ensure_ascii=True, allow_nan=False).encode("utf-8")\n\ndef strict_json(raw):\n    def pairs(items):\n        value = {}\n        for key, item in items:\n            require(key not in value, "DUPLICATE_JSON_KEY")\n            value[key] = item\n        return value\n    def reject(_):\n        raise VerificationError("NONFINITE_JSON_NUMBER")\n    try:\n        return json.loads(raw.decode("utf-8"), object_pairs_hook=pairs, parse_constant=reject)\n    except VerificationError:\n        raise\n    except (ValueError, UnicodeError, RecursionError):\n        raise VerificationError("INVALID_JSON") from None\n\ndef timestamp(value):\n    require(isinstance(value, str) and UTC.fullmatch(value) is not None, "INVALID_TIMESTAMP")\n    try:\n        return datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)\n    except ValueError:\n        raise VerificationError("INVALID_TIMESTAMP") from None\n\ndef chain_head(hashes):\n    result = sha(GENESIS)\n    for item in hashes:\n        result = sha((result + item).encode("ascii"))\n    return result\n\ndef merkle_root(hashes):\n    # The existing pulse format duplicates an odd leaf even for a one-row tree.\n    if not hashes:\n        return sha(b"empty")\n    level = [bytes.fromhex(item) for item in sorted(hashes)]\n    if len(level) % 2:\n        level.append(level[-1])\n    while len(level) > 1:\n        level = [hashlib.sha256(level[i] + level[i + 1]).digest()\n                 for i in range(0, len(level), 2)]\n        if len(level) > 1 and len(level) % 2:\n            level.append(level[-1])\n    return level[0].hex()\n\ndef verify_signature(key, signature, payload):\n    try:\n        from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey\n        from cryptography.exceptions import InvalidSignature\n    except ImportError:\n        raise Uncheckable("CRYPTOGRAPHY_REQUIRED; signature was not checked") from None\n    try:\n        Ed25519PublicKey.from_public_bytes(key).verify(signature, payload)\n    except (InvalidSignature, ValueError, TypeError):\n        raise VerificationError("INVALID_ED25519_SIGNATURE") from None\n\ndef verify_bundle(blobs, *, expected_key):\n    """Pure verification. The caller supplies its separately pinned trust anchor.\n\n    No key is accepted merely because it accompanies a signature. The network\n    entry point additionally enforces every exact source-byte digest below.\n    """\n    require(isinstance(blobs, dict) and set(blobs) == set(PINS), "WRONG_FILE_SET")\n    for name, raw in blobs.items():\n        require(type(raw) is bytes and len(raw) <= PINS[name][1], "INPUT_BYTE_LIMIT")\n    feed = strict_json(blobs["feed.json"])\n    manifest = strict_json(blobs["pulse-freeze-manifest.json"])\n    leaves = strict_json(blobs["pulse-leaves.json"])\n    require(isinstance(feed, dict) and feed.get("schema") == "csoai.gspc-pulse-feed/0.1", "FEED_SCHEMA")\n    require(isinstance(manifest, dict) and manifest.get("schema") == "csoai.pulse-freeze-manifest/0.1", "MANIFEST_SCHEMA")\n    rows = feed.get("rows")\n    require(isinstance(rows, list) and 0 < len(rows) <= 10000, "NONEMPTY_BOUNDED_ROWS_REQUIRED")\n    require(isinstance(leaves, list) and len(leaves) == len(rows), "LEAF_COUNT")\n    for count in (feed.get("row_count"), manifest.get("n_rows"), manifest.get("leaf_count")):\n        require(type(count) is int and count == len(rows), "DECLARED_COUNT")\n    times, actual = [], []\n    for row, leaf in zip(rows, leaves):\n        require(isinstance(row, dict) and isinstance(leaf, dict), "ROW_OR_LEAF_TYPE")\n        ts = row.get("ts")\n        times.append(timestamp(ts))\n        require(leaf.get("ts") == ts, "LEAF_TIMESTAMP")\n        require(isinstance(leaf.get("sha"), str) and HEX.fullmatch(leaf["sha"]) is not None, "LEAF_DIGEST")\n        try:\n            digest = sha(canonical(row))\n        except (ValueError, UnicodeError, TypeError, RecursionError):\n            raise VerificationError("ROW_CANONICALISATION") from None\n        require(hmac.compare_digest(digest, leaf["sha"]), "FEED_ROW_DIGEST")\n        actual.append(digest)\n    require(all(a < b for a, b in zip(times, times[1:])), "TIMESTAMPS_NOT_STRICTLY_ORDERED")\n    require(manifest.get("first_ts") == rows[0]["ts"], "FIRST_TIMESTAMP")\n    require(manifest.get("last_ts") == rows[-1]["ts"], "LAST_TIMESTAMP")\n    require(timestamp(manifest.get("produced_at")) >= times[-1], "FREEZE_PRECEDES_DATA")\n    require(manifest.get("chain_head") == chain_head(actual), "CHAIN_HEAD")\n    require(manifest.get("merkle_root") == merkle_root(actual), "MERKLE_ROOT")\n    require(isinstance(expected_key, str) and HEX.fullmatch(expected_key) is not None, "EXPECTED_KEY_FORMAT")\n    require(manifest.get("pubkey") == expected_key, "PINNED_KEY_MISMATCH")\n    key = bytes.fromhex(expected_key)\n    require(manifest.get("pubkey_fp") == sha(key), "KEY_FINGERPRINT")\n    try:\n        sig_text = blobs["pulse-freeze-manifest.sig"].decode("ascii").strip()\n        require(re.fullmatch(r"[0-9a-f]{128}", sig_text) is not None, "SIGNATURE_FORMAT")\n        signature = bytes.fromhex(sig_text)\n    except UnicodeError:\n        raise VerificationError("SIGNATURE_FORMAT") from None\n    verify_signature(key, signature, canonical(manifest))\n    return {"rows_verified": len(rows), "first_ts": rows[0]["ts"], "last_ts": rows[-1]["ts"],\n            "manifest_produced_at": manifest["produced_at"], "chain_head": manifest["chain_head"],\n            "merkle_root": manifest["merkle_root"], "key_fingerprint": manifest["pubkey_fp"],\n            "feed_rows_bound": True, "signature_checked": True, "source_bytes": {\n                name: {"sha256": sha(raw), "bytes": len(raw)} for name, raw in blobs.items()}}\n\ndef check_pins(blobs):\n    require(set(blobs) == set(PINS), "WRONG_FILE_SET")\n    for name, (digest, bound) in PINS.items():\n        require(type(blobs[name]) is bytes and len(blobs[name]) <= bound, "INPUT_BYTE_LIMIT")\n        require(hmac.compare_digest(sha(blobs[name]), digest), "SOURCE_PIN_MISMATCH: " + name)\n\nclass HubRedirect(urllib.request.HTTPRedirectHandler):\n    max_redirections = 3\n    max_repeats = 1\n    def redirect_request(self, req, fp, code, msg, headers, newurl):\n        parsed = urllib.parse.urlsplit(newurl)\n        require(parsed.scheme == "https" and parsed.hostname == "huggingface.co"\n                and parsed.port in (None, 443) and not parsed.username and not parsed.password,\n                "OFF_ORIGIN_REDIRECT")\n        return super().redirect_request(req, fp, code, msg, headers, newurl)\n\ndef fetch_pinned():\n    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}), HubRedirect())\n    blobs = {}\n    deadline = time.monotonic() + 60\n    for name, (_, bound) in PINS.items():\n        require(time.monotonic() < deadline, "READ_DEADLINE")\n        request = urllib.request.Request(BASE + name, headers={\n            "User-Agent": "Council-Pulse-Replay/1.0", "Accept": "application/octet-stream",\n            "Accept-Encoding": "identity"})\n        try:\n            with opener.open(request, timeout=15) as response:\n                require(response.status == 200, "HTTP_NOT_200")\n                require(response.headers.get("Content-Encoding", "identity") == "identity", "ENCODED_BODY")\n                chunks, length = [], 0\n                while True:\n                    require(time.monotonic() < deadline, "READ_DEADLINE")\n                    block = response.read1(min(65536, bound - length + 1))\n                    if not block:\n                        break\n                    length += len(block)\n                    require(length <= bound, "INPUT_BYTE_LIMIT")\n                    chunks.append(block)\n                blobs[name] = b"".join(chunks)\n        except (urllib.error.URLError, TimeoutError, OSError):\n            raise Uncheckable("PUBLIC_SOURCE_UNAVAILABLE: " + name) from None\n    check_pins(blobs)\n    return blobs\n\ndef negative_control(blobs, expected_key):\n    altered = dict(blobs)\n    feed = strict_json(blobs["feed.json"])\n    feed["rows"][0]["replay_negative_control"] = "deliberate synthetic alteration"\n    altered["feed.json"] = canonical(feed)\n    try:\n        verify_bundle(altered, expected_key=expected_key)\n    except VerificationError as error:\n        require(str(error) == "FEED_ROW_DIGEST", "NEGATIVE_CONTROL_WRONG_FAILURE")\n        return {"id": "pulse-feed-row-alteration", "expected": "FEED_ROW_DIGEST", "got": str(error)}\n    raise VerificationError("NEGATIVE_CONTROL_NOT_DETECTED")\n\ndef summary(blobs):\n    """Descriptive archive counters only; call after verify_bundle succeeds."""\n    rows = strict_json(blobs["feed.json"])["rows"]\n    valid, missing = [], 0\n    for row in rows:\n        kev = row.get("kev")\n        count = kev.get("count") if isinstance(kev, dict) else None\n        if type(count) is int and count >= 0:\n            valid.append((timestamp(row["ts"]), count))\n        else:\n            missing += 1\n    rate = None\n    seconds = (valid[-1][0] - valid[0][0]).total_seconds() if len(valid) >= 2 else 0\n    if seconds > 0:\n        rate = (valid[-1][1] - valid[0][1]) * 86400 / seconds\n    return {"scope": "descriptive replay of the pinned archive; not a current threat assessment",\n            "kev_observations": len(valid), "kev_missing_or_invalid": missing,\n            "kev_net_count_change_per_elapsed_day": rate,\n            "rate_first_ts": valid[0][0].isoformat() if valid else None,\n            "rate_last_ts": valid[-1][0].isoformat() if valid else None}\n\ndef run():\n    blobs = fetch_pinned()\n    result = verify_bundle(blobs, expected_key=PINNED_KEY)\n    result["negative_control"] = negative_control(blobs, PINNED_KEY)\n    result["archive_summary"] = summary(blobs)\n    result["checked_at"] = datetime.now(timezone.utc).isoformat()\n    result["source_repository"] = REPO\n    result["source_revision"] = REVISION\n    result["method_version"] = VERSION\n    result["model_inference_performed"] = False\n    result["training_performed"] = False\n    return result\n', '54ce532b9f605a2296d4c58ccd62dfd883551bb19640998fb5e02b7ded166c5b')

evidence_event = install_source('evidence_event', '#!/usr/bin/env python3\n# SPDX-License-Identifier: Apache-2.0\n"""csoai.evidence-event/0.1 -- one observation, derived from records that already exist.\n\nThis module never mints a measurement. It builds, canonicalises, identifies and checks events; the\nrenderers in render/ only change their shape.\n\n    python3 event.py validate EVENTS.jsonl            exit 0 when every line is a valid event\n    python3 event.py payload MEMBER EVENTS.jsonl --as-of 2026-09-30T12:00:00Z > batch.json\n                                                     the unsigned batch record that gets board-signed\n\nRules the code enforces (and the tests prove can say no):\n  * event_id = "sha256:" + sha256(RFC 8785 JCS of the event without event_id, signature, anchors).\n    Signature and anchors are attestations ABOUT the event, added later, so they are outside the id.\n  * UNMEASURED and UNCHECKABLE never carry a number: value must be null.\n  * value, when present, is a JSON number that was measured -- never an estimate, never a grade.\n  * limits[] has at least one entry.\n  * state is one of six words; there is no PASS, no score band, no "compliant".\n"""\nimport argparse, hashlib, json, math, re, sys\nfrom decimal import Decimal\n\nSCHEMA = "csoai.evidence-event/0.1"\nFABRIC_VERSION = "0.1.0"\nSTATES = ("CONSISTENT", "DIVERGENT", "PARTIAL", "UNMEASURED", "UNCHECKABLE", "NOT_DISCRIMINATING")\nNO_NUMBER_STATES = frozenset({"UNMEASURED", "UNCHECKABLE"})\nSUBJECT_KINDS = ("mcp_server", "a2a_card", "oasf_record", "model_card", "package", "policy", "scanner_report",\n                 "model_run", "signed_record", "repository", "dataset", "sandbox_run", "web_page", "route")\nID_EXCLUDE = ("event_id", "signature", "anchors")\nREQUIRED = ("schema", "subject", "claim", "method", "declared", "observed", "state", "value", "negative_control", "limits")\n# A profile narrows this schema for one kind of record (e.g. csoai.route-evidence/0.1); it never widens it.\nPROFILE_RE = re.compile(r"^csoai\\.[a-z0-9-]+/[0-9]+\\.[0-9]+$")\nBANNED_WORDS = ("certified", "certify", "compliant", "compliance status", "pass mark")\n\n\nclass DoctrineError(ValueError):\n    """An event that breaks a rule no renderer may carry forward."""\n\n\n# ---------------------------------------------------------------- RFC 8785 (JCS)\n\ndef _num(x):\n    if isinstance(x, bool):\n        raise TypeError("bool is not a number")\n    if isinstance(x, int):\n        return str(x)\n    if not math.isfinite(x):\n        raise DoctrineError("NaN/Infinity cannot be canonicalised (RFC 8785 3.2.2.3)")\n    if x == 0:\n        return "0"\n    if x.is_integer() and abs(x) < 1e21:\n        return str(int(x))\n    # ECMAScript Number.prototype.toString from Python\'s shortest round-trip repr.\n    t = Decimal(repr(abs(x))).normalize().as_tuple()\n    digits = "".join(map(str, t.digits))\n    k, n = len(digits), len(digits) + t.exponent  # n = position of the decimal point\n    sign = "-" if x < 0 else ""\n    if k <= n <= 21:\n        s = digits + "0" * (n - k)\n    elif 0 < n <= 21:\n        s = digits[:n] + "." + digits[n:]\n    elif -6 < n <= 0:\n        s = "0." + "0" * (-n) + digits\n    else:\n        ee = n - 1\n        s = digits[0] + ("." + digits[1:] if k > 1 else "") + "e" + ("+" if ee > 0 else "-") + str(abs(ee))\n    return sign + s\n\n\ndef _str(s):\n    return json.dumps(s, ensure_ascii=False)\n\n\ndef jcs(o):\n    """RFC 8785 canonical bytes. Keys sort by UTF-16 code units."""\n    def enc(v):\n        if v is None:\n            return "null"\n        if v is True:\n            return "true"\n        if v is False:\n            return "false"\n        if isinstance(v, (int, float)):\n            return _num(v)\n        if isinstance(v, str):\n            return _str(v)\n        if isinstance(v, (list, tuple)):\n            return "[" + ",".join(enc(i) for i in v) + "]"\n        if isinstance(v, dict):\n            keys = sorted(v, key=lambda k: k.encode("utf-16-be"))\n            return "{" + ",".join(_str(k) + ":" + enc(v[k]) for k in keys) + "}"\n        raise TypeError(f"not JSON: {type(v).__name__}")\n    return enc(o).encode("utf-8")\n\n\ndef sha256_hex(b):\n    return hashlib.sha256(b).hexdigest()\n\n\ndef compute_event_id(ev):\n    return "sha256:" + sha256_hex(jcs({k: v for k, v in ev.items() if k not in ID_EXCLUDE}))\n\n\ndef side_sha256(obj):\n    """sha256 of a declared/observed side, for carriers that hold a digest instead of the object."""\n    return sha256_hex(jcs(obj))\n\n\n# ---------------------------------------------------------------- build + check\n\ndef build(*, subject, claim, method, declared, observed, state, value=None, negative_control=None, limits,\n          supersedes=None, maintenance=None):\n    ev = {"schema": SCHEMA, "subject": subject, "claim": claim, "method": method, "declared": declared,\n          "observed": observed, "state": state, "value": value,\n          "negative_control": negative_control if negative_control is not None else {"id": None, "expected": None, "got": "NOT_RUN"},\n          "limits": list(limits), "supersedes": supersedes,\n          "signature": None, "anchors": {"ots": "none", "rekor": {"log_index": None, "uuid": None}},\n          "maintenance": maintenance}\n    ev["event_id"] = compute_event_id(ev)\n    errs = validate(ev)\n    if errs:\n        raise DoctrineError("; ".join(errs))\n    return ev\n\n\ndef _is_num(v):\n    return isinstance(v, (int, float)) and not isinstance(v, bool)\n\n\ndef validate(ev):\n    """Every rule, as a list of human-readable errors. Empty list = valid."""\n    e = []\n    if not isinstance(ev, dict):\n        return ["event is not an object"]\n    for k in REQUIRED:\n        if k not in ev:\n            e.append(f"missing {k}")\n    if e:\n        return e\n    if ev["schema"] != SCHEMA:\n        e.append(f"schema must be {SCHEMA}")\n    if "profile" in ev and not (isinstance(ev["profile"], str) and PROFILE_RE.match(ev["profile"])):\n        e.append("profile, when present, must name a profile such as csoai.route-evidence/0.1")\n    st = ev["state"]\n    if st not in STATES:\n        e.append(f"state {st!r} is not one of {\'/\'.join(STATES)}")\n    v = ev["value"]\n    if v is not None and not _is_num(v):\n        e.append("value must be a measured JSON number or null")\n    if st in NO_NUMBER_STATES and v is not None:\n        e.append(f"{st} carries a number ({v!r}); it must carry none")\n    if _is_num(v) and not math.isfinite(v):\n        e.append("value is not finite")\n    s = ev["subject"]\n    if not isinstance(s, dict) or s.get("kind") not in SUBJECT_KINDS or not s.get("locator"):\n        e.append("subject needs kind (known) and locator")\n    c = ev["claim"]\n    if not isinstance(c, dict) or not isinstance(c.get("text"), str) or not c.get("text"):\n        e.append("claim.text required")\n    elif c.get("source_sha256") is not None and not (isinstance(c["source_sha256"], str) and len(c["source_sha256"]) == 64):\n        e.append("claim.source_sha256 must be 64 hex or null")\n    m = ev["method"]\n    if not isinstance(m, dict) or not m.get("id") or not m.get("version"):\n        e.append("method.id and method.version required")\n    elif not (m.get("holder") == "csoai" or str(m.get("holder", "")).startswith("third_party:")):\n        e.append("method.holder must be csoai or third_party:<name>")\n    if not isinstance(ev["limits"], list) or not ev["limits"] or not all(isinstance(x, str) and x for x in ev["limits"]):\n        e.append("limits needs at least one non-empty string")\n    nc = ev["negative_control"]\n    if not isinstance(nc, dict) or "got" not in nc:\n        e.append("negative_control needs got")\n    elif nc.get("expected") is not None and nc.get("got") not in (nc.get("expected"), "NOT_RUN"):\n        e.append(f"negative control expected {nc.get(\'expected\')} but got {nc.get(\'got\')}: the method did not discriminate")\n    if st == "CONSISTENT":\n        # A CONSISTENT verdict is a claim that the method DISCRIMINATED, and it\n        # cannot be made by declaring that a control came out consistent. It has\n        # to carry: the control that was run, what that control was expected to\n        # do, and the fact that it ran and behaved that way. A record with no\n        # identified control, no expected outcome, or a control that was never\n        # run has demonstrated nothing — that is PARTIAL\'s job to say, or\n        # NOT_DISCRIMINATING\'s. (Owner invariant 4: all-pass is not competent\n        # measurement. The instrument census applies this at the instrument\n        # level; this is the same rule at the record level.)\n        nc_id = nc.get("id") if isinstance(nc, dict) else None\n        nc_expected = nc.get("expected") if isinstance(nc, dict) else None\n        nc_got = nc.get("got") if isinstance(nc, dict) else None\n        if not (isinstance(nc_id, str) and nc_id.strip()):\n            e.append("CONSISTENT needs an identified negative control (negative_control.id); "\n                     "use PARTIAL or NOT_DISCRIMINATING if no control was run")\n        if not (isinstance(nc_expected, str) and nc_expected.strip()):\n            e.append("CONSISTENT needs an expected outcome for the negative control "\n                     "(negative_control.expected); a declared outcome is not an expectation")\n        if not (isinstance(nc_got, str) and nc_got.strip()) or nc_got == "NOT_RUN":\n            e.append("CONSISTENT with no negative control run; use PARTIAL or run one")\n    blob = json.dumps({k: ev[k] for k in ("claim", "limits", "state")}, ensure_ascii=False).lower()\n    for w in BANNED_WORDS:\n        if w in blob and "never " + w not in blob and "not " + w not in blob:\n            e.append(f"doctrine word {w!r} in claim/limits")\n    if "event_id" in ev and ev["event_id"] != compute_event_id(ev):\n        e.append("event_id does not match the canonical bytes")\n    return e\n\n\ndef check_renderable(ev):\n    """Called by every renderer before it writes anything. Raises DoctrineError."""\n    errs = validate(ev)\n    if "event_id" not in ev:\n        errs.append("event has no event_id")\n    if errs:\n        raise DoctrineError("; ".join(errs))\n    return ev\n\n\ndef read_jsonl(path):\n    with open(path, encoding="utf-8") as f:\n        return [json.loads(line) for line in f if line.strip()]\n\n\ndef write_jsonl(path, events):\n    with open(path, "w", encoding="utf-8") as f:\n        for ev in events:\n            f.write(json.dumps(ev, ensure_ascii=False, sort_keys=True) + "\\n")\n\n\ndef batch_record(member, events_path, as_of, events=None):\n    """The unsigned record that /api/board-sign signs (via its sha256). Integers and strings only."""\n    raw = open(events_path, "rb").read()\n    events = events if events is not None else read_jsonl(events_path)\n    counts = {}\n    for ev in events:\n        check_renderable(ev)\n        counts[ev["state"]] = counts.get(ev["state"], 0) + 1\n    return {"schema": "csoai.evidence-batch/0.1", "as_of": as_of, "member_dir": member,\n            "events_file": {"name": events_path.rsplit("/", 1)[-1], "sha256": sha256_hex(raw), "bytes": len(raw), "n_events": len(events)},\n            "state_counts": dict(sorted(counts.items())),\n            "event_ids": [ev["event_id"] for ev in events],\n            "event_schema": SCHEMA, "fabric_version": FABRIC_VERSION,\n            "note": "Evidence, not a verdict: each event states what was declared, what was observed, and the limits of the read."}\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    sub = ap.add_subparsers(dest="cmd", required=True)\n    v = sub.add_parser("validate"); v.add_argument("events")\n    p = sub.add_parser("payload"); p.add_argument("member"); p.add_argument("events"); p.add_argument("--as-of", required=True)\n    a = ap.parse_args(argv)\n    if a.cmd == "validate":\n        bad = 0\n        for i, ev in enumerate(read_jsonl(a.events), 1):\n            errs = validate(ev) or ([] if "event_id" in ev else ["no event_id"])\n            if errs:\n                bad += 1\n                print(f"line {i}: INVALID {\'; \'.join(errs)}")\n        print(f"{\'OK\' if not bad else \'FAIL\'} {a.events}")\n        return 1 if bad else 0\n    print(json.dumps(batch_record(a.member, a.events, a.as_of), indent=1, ensure_ascii=False))\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', '144c6a36d462e21283ab9b34a6469bb32c75021da56133813d5a848412270533')

test_pulse_replay = install_source('test_pulse_replay', '# SPDX-License-Identifier: MIT\n"""Offline synthetic controls. Test keys are ephemeral, never release identities."""\nimport builtins\nimport copy\nimport json\nimport unittest\nfrom unittest.mock import patch\nfrom cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey\nfrom cryptography.hazmat.primitives.serialization import Encoding, PublicFormat\nimport pulse_replay as p\n\nclass ReplayTests(unittest.TestCase):\n    def setUp(self):\n        self.key = Ed25519PrivateKey.generate()\n        self.pub = self.key.public_key().public_bytes(Encoding.Raw, PublicFormat.Raw).hex()\n        self.rows = [{"ts": f"2026-11-03T0{i}:00:00Z", "kev": {"count": i}, "text": "café"} for i in range(3)]\n        self.blobs = self.bundle(self.rows)\n\n    def bundle(self, rows, key=None):\n        key = key or self.key\n        pub = key.public_key().public_bytes(Encoding.Raw, PublicFormat.Raw).hex()\n        leaves = [{"ts": r["ts"], "sha": p.sha(p.canonical(r))} for r in rows]\n        hashes = [l["sha"] for l in leaves]\n        manifest = {"schema": "csoai.pulse-freeze-manifest/0.1", "produced_at": "2026-11-10T00:00:00Z",\n                    "n_rows": len(rows), "leaf_count": len(rows), "first_ts": rows[0]["ts"] if rows else None,\n                    "last_ts": rows[-1]["ts"] if rows else None, "chain_head": p.chain_head(hashes),\n                    "merkle_root": p.merkle_root(hashes), "pubkey": pub, "pubkey_fp": p.sha(bytes.fromhex(pub))}\n        return {"feed.json": p.canonical({"schema": "csoai.gspc-pulse-feed/0.1", "row_count": len(rows), "rows": rows}),\n                "pulse-leaves.json": p.canonical(leaves), "pulse-freeze-manifest.json": p.canonical(manifest),\n                "pulse-freeze-manifest.sig": key.sign(p.canonical(manifest)).hex().encode()}\n\n    def verify(self, blobs=None):\n        return p.verify_bundle(self.blobs if blobs is None else blobs, expected_key=self.pub)\n\n    def change(self, filename, mutate):\n        value = json.loads(self.blobs[filename]); mutate(value); self.blobs[filename] = p.canonical(value)\n\n    def reject(self, reason):\n        with self.assertRaisesRegex(p.VerificationError, reason): self.verify()\n\n    def test_valid_odd_tree(self): self.assertEqual(self.verify()["rows_verified"], 3)\n    def test_valid_even_tree(self): self.assertEqual(self.verify(self.bundle(self.rows[:2]))["rows_verified"], 2)\n    def test_valid_single_row(self): self.assertEqual(self.verify(self.bundle(self.rows[:1]))["rows_verified"], 1)\n    def test_preserves_input_bytes(self):\n        original = dict(self.blobs); self.verify(); p.negative_control(self.blobs, self.pub); self.assertEqual(self.blobs, original)\n    def test_modified_feed_not_hidden_by_unchanged_leaves(self):\n        self.change("feed.json", lambda f: f["rows"][0]["kev"].update(count=999)); self.reject("FEED_ROW_DIGEST")\n    def test_modified_feed_and_leaf_not_hidden_by_chain(self):\n        self.change("feed.json", lambda f: f["rows"][0]["kev"].update(count=999))\n        digest = p.sha(p.canonical(json.loads(self.blobs["feed.json"])["rows"][0]))\n        self.change("pulse-leaves.json", lambda leaves: leaves[0].update(sha=digest)); self.reject("CHAIN_HEAD")\n    def test_bad_signature(self):\n        self.blobs["pulse-freeze-manifest.sig"] = b"0" * 128; self.reject("INVALID_ED25519_SIGNATURE")\n    def test_signature_required(self):\n        self.blobs["pulse-freeze-manifest.sig"] = b""; self.reject("SIGNATURE_FORMAT")\n    def test_missing_crypto_never_green(self):\n        original = builtins.__import__\n        def missing(name, *args, **kwargs):\n            if name.startswith("cryptography"): raise ImportError("synthetic unavailable dependency")\n            return original(name, *args, **kwargs)\n        with patch("builtins.__import__", side_effect=missing):\n            with self.assertRaisesRegex(p.Uncheckable, "CRYPTOGRAPHY_REQUIRED"): self.verify()\n    def test_self_selected_replacement_signer_rejected(self):\n        self.blobs = self.bundle(self.rows, Ed25519PrivateKey.generate()); self.reject("PINNED_KEY_MISMATCH")\n    def test_key_fingerprint(self):\n        self.change("pulse-freeze-manifest.json", lambda m: m.update(pubkey_fp="0"*64)); self.reject("KEY_FINGERPRINT")\n    def test_duplicate_json_key(self):\n        self.blobs["feed.json"] = b\'{"rows":[],"rows":[]}\'; self.reject("DUPLICATE_JSON_KEY")\n    def test_nan_json(self):\n        self.blobs["feed.json"] = b\'{"value":NaN}\'; self.reject("NONFINITE_JSON_NUMBER")\n    def test_invalid_utf8(self):\n        self.blobs["feed.json"] = b"\\xff"; self.reject("INVALID_JSON")\n    def test_malformed_json(self):\n        self.blobs["feed.json"] = b"{"; self.reject("INVALID_JSON")\n    def test_html_not_json(self):\n        self.blobs["feed.json"] = b"<html>fallback</html>"; self.reject("INVALID_JSON")\n    def test_duplicate_timestamps_even_with_consistent_signature(self):\n        rows = copy.deepcopy(self.rows); rows[1]["ts"] = rows[0]["ts"]\n        self.blobs = self.bundle(rows); self.reject("TIMESTAMPS_NOT_STRICTLY_ORDERED")\n    def test_reversed_order_even_with_consistent_signature(self):\n        self.blobs = self.bundle(list(reversed(self.rows))); self.reject("TIMESTAMPS_NOT_STRICTLY_ORDERED")\n    def test_timestamp_label_binding(self):\n        self.change("pulse-leaves.json", lambda leaves: leaves[0].update(ts="2026-11-01T00:00:00Z")); self.reject("LEAF_TIMESTAMP")\n    def test_invalid_calendar_date(self):\n        rows = copy.deepcopy(self.rows); rows[0]["ts"] = "2026-02-30T00:00:00Z"\n        self.blobs = self.bundle(rows); self.reject("INVALID_TIMESTAMP")\n    def test_leaf_count(self):\n        self.change("pulse-leaves.json", lambda leaves: leaves.pop()); self.reject("LEAF_COUNT")\n    def test_boolean_count_is_not_one(self):\n        self.blobs = self.bundle(self.rows[:1]); self.change("feed.json", lambda f: f.update(row_count=True)); self.reject("DECLARED_COUNT")\n    def test_wrong_manifest_count(self):\n        self.change("pulse-freeze-manifest.json", lambda m: m.update(n_rows=4)); self.reject("DECLARED_COUNT")\n    def test_first_timestamp(self):\n        self.change("pulse-freeze-manifest.json", lambda m: m.update(first_ts=self.rows[-1]["ts"])); self.reject("FIRST_TIMESTAMP")\n    def test_last_timestamp(self):\n        self.change("pulse-freeze-manifest.json", lambda m: m.update(last_ts=self.rows[0]["ts"])); self.reject("LAST_TIMESTAMP")\n    def test_freeze_cannot_precede_its_rows(self):\n        self.change("pulse-freeze-manifest.json", lambda m: m.update(produced_at="2026-11-01T00:00:00Z")); self.reject("FREEZE_PRECEDES_DATA")\n    def test_merkle_root(self):\n        self.change("pulse-freeze-manifest.json", lambda m: m.update(merkle_root="0"*64)); self.reject("MERKLE_ROOT")\n    def test_empty_snapshot_is_not_verified(self):\n        self.blobs = self.bundle([]); self.reject("NONEMPTY_BOUNDED_ROWS_REQUIRED")\n    def test_missing_file(self):\n        self.blobs.pop("feed.json"); self.reject("WRONG_FILE_SET")\n    def test_extra_file(self):\n        self.blobs["other.json"] = b"{}"; self.reject("WRONG_FILE_SET")\n    def test_oversized_input(self):\n        self.blobs["feed.json"] = b" "*(p.PINS["feed.json"][1]+1); self.reject("INPUT_BYTE_LIMIT")\n    def test_immutable_network_pins_reject_other_valid_snapshot(self):\n        with self.assertRaisesRegex(p.VerificationError, "SOURCE_PIN_MISMATCH"): p.check_pins(self.blobs)\n    def test_negative_control_is_actually_executed(self):\n        self.assertEqual(p.negative_control(self.blobs, self.pub)["got"], "FEED_ROW_DIGEST")\n    def test_absent_counts_stay_absent(self):\n        rows = [{"ts": r["ts"], "kev": None} for r in self.rows]\n        result = p.summary(self.bundle(rows)); self.assertEqual(result["kev_missing_or_invalid"], 3)\n        self.assertIsNone(result["kev_net_count_change_per_elapsed_day"])\n    def test_zero_count_not_missing(self): self.assertEqual(p.summary(self.blobs)["kev_observations"], 3)\n    def test_same_day_elapsed_time_not_floored_to_day(self):\n        self.assertEqual(p.summary(self.blobs)["kev_net_count_change_per_elapsed_day"], 24)\n    def test_single_observation_has_no_rate(self):\n        self.assertIsNone(p.summary(self.bundle(self.rows[:1]))["kev_net_count_change_per_elapsed_day"])\n    def test_redirect_off_origin_rejected(self):\n        req = p.urllib.request.Request(p.BASE+"feed.json")\n        for target in ("http://huggingface.co/a", "https://example.com/a", "https://huggingface.co@127.0.0.1/a"):\n            with self.subTest(target=target):\n                with self.assertRaisesRegex(p.VerificationError, "OFF_ORIGIN_REDIRECT"):\n                    p.HubRedirect().redirect_request(req, None, 302, "", {}, target)\n    def test_public_source_unavailable_is_not_success(self):\n        class Broken:\n            def open(self, *args, **kwargs): raise p.urllib.error.URLError("offline")\n        with patch.object(p.urllib.request, "build_opener", return_value=Broken()):\n            with self.assertRaisesRegex(p.Uncheckable, "PUBLIC_SOURCE_UNAVAILABLE"): p.fetch_pinned()\n\nif __name__ == "__main__":\n    unittest.main()\n', '5df67dbd2481900615a58c90cf0f6f4120cdb84add6302eefb3864db0009218b')

SOURCE_LOCK = {'method_version': '1.0.0', 'source_files': {'pulse_replay.py': {'sha256': '54ce532b9f605a2296d4c58ccd62dfd883551bb19640998fb5e02b7ded166c5b', 'bytes': 11839}, 'test_pulse_replay.py': {'sha256': '5df67dbd2481900615a58c90cf0f6f4120cdb84add6302eefb3864db0009218b', 'bytes': 9051}, 'evidence_event.py': {'sha256': '144c6a36d462e21283ab9b34a6469bb32c75021da56133813d5a848412270533', 'bytes': 12186}}, 'expected_tests': 39, 'event_helper_source': {'repository': 'CSOAI-ORG/councilof-ai', 'revision': '3bfcac428e3b48e0fc99f5aca5cd40ee181e317e', 'path': 'packages/evidence-fabric/event.py', 'sha256': '144c6a36d462e21283ab9b34a6469bb32c75021da56133813d5a848412270533'}, 'data_scope': 'Pinned public archive, 23 observations through 2026-11-09; not a live measurement or model benchmark', 'gpu': False, 'tpu': False, 'model_inference': False, 'training': False}


## 2. Run the offline regression controls
An empty, incomplete or skipped suite cannot satisfy this notebook's qualification.

In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromModule(test_pulse_replay)
if suite.countTestCases() != 39:
    raise RuntimeError("TEST_INVENTORY_CHANGED")
stream = io.StringIO()
result = unittest.TextTestRunner(stream=stream, verbosity=1).run(suite)
print(stream.getvalue())
if (not result.wasSuccessful() or result.testsRun != 39
        or result.skipped or result.expectedFailures or result.unexpectedSuccesses):
    raise RuntimeError("REGRESSION_QUALIFICATION_FAILED")
TEST_RECEIPT = {"tests_run": result.testsRun, "failures": len(result.failures),
                "errors": len(result.errors), "skipped": len(result.skipped),
                "scope": "offline synthetic controls; not model benchmark cases",
                "source_files": SOURCE_LOCK["source_files"], "python": platform.python_version(),
                "platform": platform.system(), "architecture": platform.machine()}
print(json.dumps(TEST_RECEIPT, indent=2))


## 3. Replay the immutable public snapshot
Successful execution writes three downloadable output JSON files. The event uses the existing `csoai.evidence-event/0.1` contract, has a null value and no signature or execution authority. Failures stop before a successful event is created.

In [ ]:
# Four fixed, public, immutable HF URLs; no private data, model calls or signing.
REPLAY = pulse_replay.run()
EVENT = evidence_event.build(
    subject={"kind": "dataset", "locator": pulse_replay.BASE + "feed.json", "declared_by": "Council of AI"},
    claim={"text": "The pinned archive's actual feed rows reproduce its retained signed freeze under the source-pinned lane key.",
           "source_url": pulse_replay.BASE + "pulse-freeze-manifest.json",
           "source_sha256": REPLAY["source_bytes"]["pulse-freeze-manifest.json"]["sha256"],
           "read_at": REPLAY["checked_at"]},
    method={"id": "pulse-archive-row-binding-replay", "version": pulse_replay.VERSION, "holder": "csoai",
            "code_sha256": SOURCE_LOCK["source_files"]["pulse_replay.py"]["sha256"]},
    declared={"source_revision": pulse_replay.REVISION, "expected_source_hashes": {k: v[0] for k, v in pulse_replay.PINS.items()},
              "pinned_lane_public_key": pulse_replay.PINNED_KEY},
    observed={"replay": REPLAY, "regression_tests": TEST_RECEIPT},
    state="CONSISTENT", value=None, negative_control=REPLAY["negative_control"],
    limits=[
        "This is a replay of a fixed historical public archive, not fresh data collection or a model capability result.",
        "The lane key is pinned to source history, not independently authenticated issuer identity and not the GSPC DID signing key.",
        "Signature validity and row binding establish byte consistency, not the factual truth of the underlying observations.",
        "No model inference, training, human approval, execution authority, measurement admission or certification is established.",
        "Synthetic controls are instrument checks, not additional real-world observations."])
if evidence_event.validate(EVENT):
    raise RuntimeError("CANONICAL_EVENT_VALIDATION_FAILED")
for filename, value in [("pulse-replay-result.json", REPLAY), ("pulse-replay-event.json", EVENT), ("test-results.json", TEST_RECEIPT)]:
    Path(filename).write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")
print(json.dumps({"state": EVENT["state"], "event_id": EVENT["event_id"],
                  "rows_verified": REPLAY["rows_verified"], "signature_checked": REPLAY["signature_checked"],
                  "negative_control": REPLAY["negative_control"], "data_last_observed": REPLAY["last_ts"],
                  "checked_at": REPLAY["checked_at"], "training": False, "inference": False}, indent=2))


## Reading the result
`CONSISTENT` applies only to this snapshot's byte binding and the stated controls. It is not a current health report, an endorsement, model benchmark, safety determination or a claim that the original observations were true. Read `pulse-replay-event.json` for the exact inputs, method, actual test result and limitations.